# XGBoost preprocessing utilities

This notebook contains the reusable preprocessing and pipeline-construction logic used by `xgboost_training.ipynb`.

It is intentionally separated so preprocessing and training can be maintained independently.

## Exposed helpers

- `prepare_raw_features`
- `align_features_for_pipeline`
- `build_xgb_pipeline`
- `transform_features_before_classifier`
- `feature_importance_series`

In [ ]:
from typing import Any

import numpy as np
import pandas as pd
from pandas import CategoricalDtype
from pandas.api.types import is_string_dtype
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectPercentile, VarianceThreshold, f_classif
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier


def _text_like_column_names(X: pd.DataFrame) -> list[str]:
    out: list[str] = []
    for c in X.columns:
        dt = X[c].dtype
        if dt == object or dt == np.dtype("O"):
            out.append(c)
        elif isinstance(dt, CategoricalDtype) or is_string_dtype(X[c]):
            out.append(c)
    return out


def _coerce_numeric_like_text_columns(
    X: pd.DataFrame,
    *,
    min_numeric_ratio: float = 0.85,
) -> pd.DataFrame:
    """Convert object/string columns to numeric when most values are numeric-like."""
    X = X.copy()
    text_cols = _text_like_column_names(X)
    for c in text_cols:
        s = X[c]
        as_str = s.astype("string")
        cleaned = as_str.str.replace(",", "", regex=False).str.strip()
        parsed = pd.to_numeric(cleaned, errors="coerce")

        non_null = int(s.notna().sum())
        if non_null == 0:
            continue

        numeric_like = int(parsed.notna().sum())
        ratio = numeric_like / non_null
        if ratio >= min_numeric_ratio:
            X[c] = parsed.astype(float)
    return X


def prepare_raw_features(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()

    # Step 1: promote numeric-like text columns to real numeric features.
    X = _coerce_numeric_like_text_columns(X, min_numeric_ratio=0.85)

    # Step 2: force all numeric columns through robust parser.
    for c in X.select_dtypes(include=[np.number]).columns:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    # Step 3: remaining text-like columns stay as strings for OHE.
    for c in _text_like_column_names(X):
        X[c] = X[c].map(lambda v: v if pd.isna(v) else str(v))
    return X


def infer_prep_column_groups(prep: ColumnTransformer) -> tuple[list[str], list[str]]:
    num_cols: list[str] = []
    cat_cols: list[str] = []
    for name, _trans, cols in prep.transformers_:
        if name in ("remainder", "drop"):
            continue
        if name == "num":
            num_cols = list(cols)
        elif name == "cat":
            cat_cols = list(cols)
    return num_cols, cat_cols


def align_features_for_pipeline(model: Pipeline, X: pd.DataFrame) -> pd.DataFrame:
    prep = model.named_steps["prep"]
    num_cols, cat_cols = infer_prep_column_groups(prep)
    X = X.copy()
    for c in num_cols:
        if c in X.columns:
            X[c] = pd.to_numeric(X[c], errors="coerce")
    for c in cat_cols:
        if c in X.columns:
            X[c] = X[c].map(lambda v: v if pd.isna(v) else str(v))
    return X


def _one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def _column_transformer(**kwargs) -> ColumnTransformer:
    try:
        return ColumnTransformer(n_jobs=1, sparse_output=True, **kwargs)
    except TypeError:
        try:
            return ColumnTransformer(n_jobs=1, sparse_threshold=1.0, **kwargs)
        except TypeError:
            return ColumnTransformer(n_jobs=1, **kwargs)


def build_xgb_pipeline(
    X: pd.DataFrame,
    *,
    select_percentile: float = 40.0,
    variance_threshold: float = 1e-8,
    random_state: int = 42,
    scale_pos_weight: float | None = None,
    xgb_params: dict[str, Any] | None = None,
) -> Pipeline:
    numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
    categorical_features = X.select_dtypes(include=["object", "category", "bool", "string"]).columns.tolist()

    numeric_transformer = Pipeline(
        steps=[
            # add_indicator adds binary missingness flags, often predictive in credit data.
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ]
    )
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", _one_hot_encoder()),
        ]
    )

    preprocessor = _column_transformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ],
        remainder="drop",
    )

    clf_kw: dict[str, Any] = dict(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        min_child_weight=2,
        reg_lambda=1.0,
        random_state=random_state,
        n_jobs=-1,
        eval_metric="logloss",
        tree_method="hist",
    )
    if scale_pos_weight is not None:
        clf_kw["scale_pos_weight"] = float(scale_pos_weight)
    if xgb_params:
        clf_kw.update(xgb_params)

    return Pipeline(
        steps=[
            ("prep", preprocessor),
            ("var", VarianceThreshold(threshold=variance_threshold)),
            ("select", SelectPercentile(score_func=f_classif, percentile=select_percentile)),
            ("clf", XGBClassifier(**clf_kw)),
        ]
    )


def transform_features_before_classifier(model: Pipeline, X: pd.DataFrame) -> Any:
    Xt: Any = X
    for _name, step in model.steps[:-1]:
        Xt = step.transform(Xt)
    return Xt


def feature_importance_series(model: Pipeline) -> pd.Series:
    clf = model.named_steps["clf"]
    prep = model.named_steps["prep"]
    var = model.named_steps["var"]
    sel = model.named_steps["select"]
    names = prep.get_feature_names_out()
    names = names[var.get_support()]
    names = names[sel.get_support()]
    if len(names) != len(clf.feature_importances_):
        return pd.Series(clf.feature_importances_)
    return pd.Series(clf.feature_importances_, index=names).sort_values(ascending=False)


def preprocess_and_save_train_data(
    train_path,
    *,
    target_col: str = "Default",
    out_path=None,
) -> pd.DataFrame:
    """Load raw train CSV, apply preprocessing, save processed CSV, return dataframe."""
    df = pd.read_csv(train_path, low_memory=False)
    if target_col not in df.columns:
        raise ValueError(f"Missing target column: {target_col}")

    X = prepare_raw_features(df.drop(columns=[target_col]))
    y = df[target_col].astype(int)
    processed = X.copy()
    processed[target_col] = y

    if out_path is not None:
        out_path = str(out_path)
        import os

        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        processed.to_csv(out_path, index=False)

    return processed


print("Preprocessing helpers ready.")